***RAG***
1. Install dependencies
    - Transformers / Sentence-transformers
    - PyMuPdf
    - FAISS
2. Load pre-trained model (llama-3-Korean-Bllossom-8B from https://huggingface.co/MLP-KTLim/llama-3-Korean-Bllossom-8B)
3. Preprocessing data

***1. Install Libraries***

In [ ]:
!pip install transformers==4.40.0 accelerate datasets sentence-transformers
!pip install pyPDF2
!pip install pymupdf
!pip install faiss-gpu

***2. Load Pre-trained Model***

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("MLP-KTLim/llama-3-Korean-Bllossom-8B")
model = AutoModelForCausalLM.from_pretrained("MLP-KTLim/llama-3-Korean-Bllossom-8B",
                                             torch_dtype=torch.bfloat16,
                                             device_map="auto")

model.eval()

***3. Preprocessing Data***

In [ ]:
# simple pdf file to test
pdf_path = "test.pdf"

Extract text from pdf

In [ ]:
import fitz

def extract_text_from_pdf(file_path):
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

raw_text = extract_text_from_pdf(pdf_path)
print(raw_text)

Clean extracted text

In [ ]:
import re

def clean_text(text):
    text = re.sub(r'\s+', ' ', text) # 공백 중복 처리
    text = re.sub(r'[^\w\s]', '', text) # 특수문자 제거
    return text.strip()

cleaned_text = clean_text(raw_text)
print(cleaned_text)

Split text into chunks

In [ ]:
def split_text_into_chunks(text, chunk_size=200, overlap_size=20):
    words = text.split()
    #chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size-overlap_size)]
    return chunks

chunks = split_text_into_chunks(cleaned_text)

print(chunks)
print(len(chunks))

Vectorize chunks

In [ ]:
from sentence_transformers import SentenceTransformer

retriever_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = retriever_model.encode(chunks)

In [ ]:
import faiss
import numpy as np

def create_faiss_index(chunks):

    # FAISS Index
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(np.array(embeddings))

    return index, embeddings

index, embeddings = create_faiss_index(chunks)